# 🍃 V53: THE LEAF-WISE ASSASSIN (Local LightGBM 3-in-1 Ensemble) 🍃\nKita berpindah ke rajanya kompetisi time-series: **LightGBM**. Dengan penumbuhan cabang secara asimetris (*Leaf-Wise*), model ini tidak akan bingung membedakan genangan 1 meter dengan banjir ekstrem 200 meter.\n\nSetiap stasiun akan dilatih menggunakan **3 Arsitektur Terbaik** (Top 3 Optuna Trials) dan ditebak secara rata-rata untuk membunuh *variance error* hingga tuntas!

In [ ]:
!pip install -q lightgbm optuna
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')

if not 'path' in locals():
    if os.path.exists('/kaggle/input'):
        dataset_folders = [f for f in os.listdir('/kaggle/input') if os.path.isdir(f'/kaggle/input/{f}')]
        path = f'/kaggle/input/{dataset_folders[0]}/' if dataset_folders else '/kaggle/input/'
    elif os.path.exists('/content'):
        from google.colab import drive
        drive.mount('/content/drive')
        path = '/content/drive/MyDrive/SSF-SSDS/'
    else:
        path = './'

train_df = pd.read_csv(path + 'train.csv', parse_dates=['datetime'])
test_df = pd.read_csv(path + 'test.csv')
lingkungan_df = pd.read_csv(path + 'data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])
sample_sub = pd.read_csv(path + 'sample_submission.csv')
coords_df = pd.read_csv(path + 'data_pendukung/koordinat_pos.csv')

print("Data berhasil dimuat!")

## 1. Pemetaan Hulu-Hilir (The Water Network)

In [ ]:
print("Membangun Peta Aliran...")
elevations = train_df.groupby('nama_pos')['tma_mdpl'].mean().to_dict()
coords = coords_df.copy()
coords['elevation'] = coords['nama_pos'].map(elevations)

upstream_map = {pos: [] for pos in coords['nama_pos']}

for i, row_i in coords.iterrows():
    pos_i = row_i['nama_pos']
    elev_i = row_i['elevation']
    
    candidates = coords[coords['elevation'] < elev_i].copy()
    if len(candidates) > 0:
        dist = np.sqrt((candidates['latitude'] - row_i['latitude'])**2 + (candidates['longitude'] - row_i['longitude'])**2)
        nearest_idx = dist.idxmin()
        pos_hilir = coords.loc[nearest_idx, 'nama_pos']
        upstream_map[pos_hilir].append(pos_i)

for pos in list(upstream_map.keys())[:5]:
    print(f"[{pos}] Hulu: {upstream_map[pos]}")

## 2. Fitur Lingkungan & Fisika Tanah (EWM)

In [ ]:
print("Mengekstraksi Fisika Tanah & Cuaca...")
lingkungan_df = lingkungan_df.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

def build_env_features(env_df):
    agg_df = env_df[['datetime', 'nama_pos']].copy()
    grouped = env_df.groupby('nama_pos')
    
    agg_df['rain_sum_72h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(72, min_periods=1).sum())
    agg_df['rain_sum_168h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(168, min_periods=1).sum())
    agg_df['rain_ewm_48h'] = grouped['rainfall_mm'].transform(lambda x: x.ewm(halflife=48, ignore_na=True).mean())
    
    if 'soil_moisture_100_255cm' in env_df.columns:
        agg_df['deep_soil_ewm_168h'] = grouped['soil_moisture_100_255cm'].transform(lambda x: x.ewm(halflife=168, ignore_na=True).mean())
        agg_df['soil_0_7cm_sum_72h'] = grouped['soil_moisture_0_7cm'].transform(lambda x: x.rolling(72, min_periods=1).sum())
        
    return agg_df

env_feat = build_env_features(lingkungan_df)

## 3. The Time-Machine (Multi-Lag Upstream Injection)

In [ ]:
def prepare_dataset_multi_lag(base_df, env_feat, historical_tma_df, upstream_map):
    df = base_df.copy()
    
    df['hour'] = df['datetime'].dt.hour
    df['month'] = df['datetime'].dt.month
    df['dayofyear'] = df['datetime'].dt.dayofyear
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24.0)
    
    df = pd.merge(df, env_feat, on=['datetime', 'nama_pos'], how='left')
    
    historical_map = historical_tma_df.set_index(['nama_pos', 'datetime'])['tma_mdpl'].to_dict()
    df['tma_lag_365d'] = df.apply(lambda row: historical_map.get((row['nama_pos'], row['datetime'] - pd.DateOffset(years=1)), np.nan), axis=1)
    
    last_known_map = historical_tma_df.dropna(subset=['tma_mdpl']).sort_values('datetime').groupby('nama_pos').tail(1).set_index('nama_pos')['tma_mdpl'].to_dict()
    df['last_known_tma'] = df['nama_pos'].map(last_known_map)
    
    print("  -> Menyuntikkan Kaskade Hulu dengan Waktu Tempuh (Multi-Lag)...")
    upstream_dfs = []
    
    for pos, upstreams in upstream_map.items():
        if len(upstreams) > 0:
            up_df = df[df['nama_pos'].isin(upstreams)]
            agg = up_df.groupby('datetime')[['rain_sum_168h', 'rain_sum_72h']].mean().reset_index()
        else:
            up_df = df[df['nama_pos'] == pos]
            agg = up_df[['datetime', 'rain_sum_168h', 'rain_sum_72h']].copy()
            
        agg = agg.rename(columns={
            'rain_sum_168h': 'upstream_rain_sum_168h',
            'rain_sum_72h': 'upstream_rain_sum_72h'
        })
        
        # 🕰️ MULTI-LAG THE TIME-MACHINE 🕰️
        agg['upstream_rain_168h_lag_1d'] = agg['upstream_rain_sum_168h'].shift(24) 
        agg['upstream_rain_168h_lag_2d'] = agg['upstream_rain_sum_168h'].shift(48)
        agg['upstream_rain_168h_lag_3d'] = agg['upstream_rain_sum_168h'].shift(72)
        
        agg['nama_pos'] = pos
        upstream_dfs.append(agg)
        
    all_upstream = pd.concat(upstream_dfs, ignore_index=True)
    df = pd.merge(df, all_upstream, on=['datetime', 'nama_pos'], how='left')
    
    return df

print("Menyusun Train & Test Set (Local Time-Machine)...")
historical_tma = train_df[['datetime', 'nama_pos', 'tma_mdpl']].copy()

train_full = prepare_dataset_multi_lag(train_df, env_feat, historical_tma, upstream_map)

test_parsed = test_df.copy()
split_info = test_parsed['id'].str.split(' - ', n=1, expand=True)
test_parsed['datetime'] = pd.to_datetime(split_info[0])
test_parsed['nama_pos'] = split_info[1]
test_full = prepare_dataset_multi_lag(test_parsed, env_feat, historical_tma, upstream_map)

for st in train_full['nama_pos'].unique():
    mask_tr = train_full['nama_pos'] == st
    train_full.loc[mask_tr, 'tma_lag_365d'] = train_full.loc[mask_tr, 'tma_lag_365d'].fillna(train_full.loc[mask_tr, 'last_known_tma'])
    mask_te = test_full['nama_pos'] == st
    test_full.loc[mask_te, 'tma_lag_365d'] = test_full.loc[mask_te, 'tma_lag_365d'].fillna(test_full.loc[mask_te, 'last_known_tma'])

train_full = train_full.dropna(subset=['tma_mdpl']).reset_index(drop=True)

drop_cols = ['datetime', 'nama_pos', 'tma_mdpl', 'id']
features = [c for c in train_full.columns if c not in drop_cols]
print(f"Total Fitur: {len(features)}")


## 4. Pelatihan 30 Model Lokal x 3 Ensemble (Top-3 LightGBM)

In [ ]:
stations = train_full['nama_pos'].unique()
station_models = {}
station_scores = {}

optuna.logging.set_verbosity(optuna.logging.ERROR)
N_TRIALS = 15

print(f"Memulai Optuna LightGBM (Tweedie Asimetris) untuk 30 Stasiun...\n")

for st in stations:
    X_st = train_full[train_full['nama_pos'] == st][features]
    y_st = train_full[train_full['nama_pos'] == st]['tma_mdpl']
    
    def objective(trial):
        params = {
            'objective': 'tweedie', 
            'tweedie_variance_power': trial.suggest_float('tweedie_variance_power', 1.1, 1.9),
            'metric': 'rmse', 
            'boosting_type': 'gbdt',
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 63), # Leaf-Wise Growth!
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': 1,
            'verbose': -1,
            'random_state': 42
        }
        
        split_idx = int(len(X_st) * 0.8)
        X_tr, y_tr = X_st.iloc[:split_idx], np.clip(y_st.iloc[:split_idx], 0.0001, None)
        X_va, y_va = X_st.iloc[split_idx:], np.clip(y_st.iloc[split_idx:], 0.0001, None)
        
        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval = lgb.Dataset(X_va, label=y_va, reference=dtrain)
        
        callbacks = [lgb.early_stopping(30, verbose=False)]
        model = lgb.train(params, dtrain, valid_sets=[dval], num_boost_round=800, callbacks=callbacks)
        
        preds = model.predict(X_va)
        return np.sqrt(mean_squared_error(y_va, preds))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS)
    
    # 🌟 AMBIL 3 TERBAIK UNTUK DI-ENSEMBLE LOKAL 🌟
    top_trials = sorted(study.trials, key=lambda t: t.value)[:3]
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    models = []
    
    for i, trial in enumerate(top_trials):
        params = trial.params.copy()
        params.update({'objective': 'tweedie', 'metric': 'rmse', 'verbose': -1, 'random_state': 42+i})
        
        # Train di satu fold saja sebagai representasi model
        tr_idx, va_idx = next(kf.split(X_st))
        X_tr, y_tr = X_st.iloc[tr_idx], np.clip(y_st.iloc[tr_idx], 0.0001, None)
        X_va, y_va = X_st.iloc[va_idx], np.clip(y_st.iloc[va_idx], 0.0001, None)
        
        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval = lgb.Dataset(X_va, label=y_va, reference=dtrain)
        
        callbacks = [lgb.early_stopping(50, verbose=False)]
        model = lgb.train(params, dtrain, valid_sets=[dval], num_boost_round=1000, callbacks=callbacks)
        
        models.append(model)
        
    # Hitung validasi rata-rata ensemble (di sisa data 20%)
    split_idx = int(len(X_st) * 0.8)
    X_test_val = X_st.iloc[split_idx:]
    y_test_val = np.clip(y_st.iloc[split_idx:], 0.0001, None)
    
    ensemble_preds = np.zeros(len(X_test_val))
    for m in models:
        ensemble_preds += m.predict(X_test_val) / 3.0
        
    avg_rmse = np.sqrt(mean_squared_error(y_test_val, ensemble_preds))
    
    station_models[st] = models
    station_scores[st] = avg_rmse
    
    print(f"[{st}] - 3-in-1 Ensemble RMSE: {avg_rmse:.4f}")

global_rmse = np.mean(list(station_scores.values()))
print(f"\n🏆 GRAND AVERAGE RMSE ACROSS 30 STATIONS: {global_rmse:.4f}")

## 5. Visualisasi Asimetris & Submission

In [ ]:
TARGET_STATION = "Napel" # Stasiun dengan anomali 294 meter

fig, ax = plt.subplots(1, 2, figsize=(22, 8))

# Ambil model pertama dari stasiun target
imp = station_models[TARGET_STATION][0].feature_importance(importance_type='gain')
imp_df = pd.DataFrame({'Features': features, 'Importance score': imp})
imp_df = imp_df.sort_values(by='Importance score', ascending=False).head(20)

sns.barplot(x='Importance score', y='Features', data=imp_df, ax=ax[0], palette='viridis')
ax[0].set_title(f'LightGBM Gain Importance [{TARGET_STATION}]')

X_target = train_full[train_full['nama_pos'] == TARGET_STATION][features]
y_target = train_full[train_full['nama_pos'] == TARGET_STATION]['tma_mdpl']

val_preds = np.zeros(len(X_target))
for m in station_models[TARGET_STATION]:
    val_preds += m.predict(X_target) / 3.0

sub_actual = y_target.iloc[-200:].values
sub_pred = val_preds[-200:]
ax[1].plot(sub_actual, label='Actual TMA', color='blue', alpha=0.7, linewidth=2)
ax[1].plot(sub_pred, label='Predicted (LGBM Leaf-Wise)', color='red', alpha=0.7, linewidth=2)
ax[1].set_title(f'Actual vs Predicted - 200 Hari Terakhir [{TARGET_STATION}]')
ax[1].legend()

plt.tight_layout()
plt.show()

# ================= BAGIAN TEST SET =================
test_preds = np.zeros(len(test_full))

for st in stations:
    idx = test_full['nama_pos'] == st
    if idx.sum() == 0: continue
        
    X_test_st = test_full.loc[idx, features]
    
    st_preds = np.zeros(len(X_test_st))
    for m in station_models[st]:
        st_preds += m.predict(X_test_st) / 3.0
        
    test_full.loc[idx, 'tma_mdpl'] = st_preds

sub_pred = test_full[['id', 'tma_mdpl']].copy()
sub = pd.merge(sample_sub[['id']], sub_pred, on='id', how='left')
sub['tma_mdpl'] = sub['tma_mdpl'].fillna(method='ffill').fillna(method='bfill')

output_file = 'submission_V53_Local_LightGBM.csv'
sub.to_csv(output_file, index=False)
print(f"\n✅ Selesai! File '{output_file}' siap disubmit. The Leaf-Wise Assassin telah beraksi!")